In [1]:
import numpy as np
from scipy.stats import norm

In [2]:
n = 100
teta = 10

beta = 0.95

In [3]:
def calc_x(y, teta):
    return (1 / (1 - y)) ** (1 / (teta - 1))

def create_sample():
    answ = []
    for _ in range(n):
        y = np.random.uniform(0, 1)
        x = calc_x(y, teta)
        answ.append(x)
    return np.array(answ)

In [4]:
sample = create_sample()

In [5]:
print(sample.max())

2.2348034862173947


In [6]:
def calc_teta_mark(sample):
    return 1 + n / np.sum(np.log(sample))

In [7]:
teta_mark = calc_teta_mark(sample)

In [8]:
def calc_med_by_teta(teta):
    return np.pow(2, 1 / (teta - 1))

In [18]:
t1 = norm.ppf(0.025)
t2 = norm.ppf(0.975)


sigma_teta_mark = np.pow(2, 1 / (teta_mark - 1)) * np.log(2) / (teta_mark - 1)

left_bound = calc_med_by_teta(teta_mark) - t2 * sigma_teta_mark / np.sqrt(n)
right_bound = calc_med_by_teta(teta_mark) - t1 * sigma_teta_mark / np.sqrt(n)

print(f"точное значение медианы: {calc_med_by_teta(teta)}")
print(f"доверительный интервал для медианы: {left_bound} < median < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

точное значение медианы: 1.080059738892306
доверительный интервал для медианы: 1.0697162886791278 < median < 1.1055257230861204
его длина: 0.03580943440699258


In [23]:
sigma_teta_mark = teta_mark - 1

left_bound = teta_mark - t2 * sigma_teta_mark / np.sqrt(n)
right_bound = teta_mark - t1 * sigma_teta_mark / np.sqrt(n)

print(f"teta = {teta}")
print(f"доверительный интервал для параметра распределения: {left_bound} < teta < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

teta = 10
доверительный интервал для параметра распределения: 7.635011304062398 < teta < 10.869918005468387
его длина: 3.234906701405989


In [11]:
def do_bootstrap(cnt: int, sample: np.ndarray, param_func) -> np.ndarray:
    """
    Здесь просто генерация выборок
    """
    answ = []
    for i in range(cnt):
        podviborka = []
        for j in range(sample.size):
            index = np.random.randint(0, n)
            podviborka.append(sample[index])
        podviborka = np.array(podviborka)
        answ.append(param_func(podviborka))
    return np.array(answ)

In [12]:
def do_parametric_bootstrap(cnt: int, sample: np.ndarray, param_func) -> np.ndarray:
    """
    Здесь просто генерация выборок
    """
    answ = []
    for i in range(cnt):
        podviborka = []
        for j in range(sample.size):
            val = calc_x(np.random.uniform(0, 1), teta_mark)
            podviborka.append(val)
        podviborka = np.array(podviborka)
        answ.append(param_func(podviborka))
    return np.array(answ)

In [13]:
def auxiliary_func(sample):
    teta = calc_teta_mark(sample)
    return calc_med_by_teta(teta)

In [29]:
boots = do_bootstrap(1000, sample, auxiliary_func)

sample_med = calc_med_by_teta(teta_mark)

deltas = boots - sample_med

k1 = 25
k2 = 975

var_series = sorted(deltas)

left_bound = sample_med - var_series[k2]
right_bound = sample_med - var_series[k1]

print(f"доверительный интервал для медианы (непараметрический бутстрап)\n: {left_bound} < median < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

доверительный интервал для медианы (непараметрический бутстрап)
: 1.0669906658606987 < median < 1.1048511472582512
его длина: 0.03786048139755249


In [25]:
boots = do_parametric_bootstrap(50000, sample, auxiliary_func)

sample_med = calc_med_by_teta(teta_mark)

deltas = boots - sample_med

k1 = int((1 - beta) / 2 * 50000)
k2 = int((1 + beta) / 2 * 50000)

var_series = sorted(deltas)

left_bound = sample_med - var_series[k2]
right_bound = sample_med - var_series[k1]

print(f"доверительный интервал для медианы (параметрический бутстрап)\n: {left_bound} < median < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

доверительный интервал для медианы (параметрический бутстрап)
: 1.0688531384076212 < median < 1.1043972265781874
его длина: 0.0355440881705662


In [27]:
boots = do_bootstrap(1000, sample, calc_teta_mark)

deltas = boots - teta_mark

k1 = 25
k2 = 975

var_series = sorted(deltas)

left_bound = teta_mark - var_series[k2]
right_bound = teta_mark - var_series[k1]

print(f"доверительный интервал для параметра распределения (непараметрический бутстрап)\n: {left_bound} < teta < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

доверительный интервал для параметра распределения (непараметрический бутстрап)
: 7.019907455775252 < teta < 10.716807035601558
его длина: 3.6968995798263062


In [28]:
boots = do_parametric_bootstrap(50000, sample, calc_teta_mark)

deltas = boots - teta_mark

k1 = int((1 - beta) / 2 * 50000)
k2 = int((1 + beta) / 2 * 50000)

var_series = sorted(deltas)

left_bound = teta_mark - var_series[k2]
right_bound = teta_mark - var_series[k1]

print(f"доверительный интервал для параметра распределения (парамтрический бутстрап)\n: {left_bound} < teta < {right_bound}")
print(f"его длина: {right_bound - left_bound}")

доверительный интервал для параметра распределения (парамтрический бутстрап)
: 7.355363320205578 < teta < 10.672993129841597
его длина: 3.317629809636019
